## OpenAI 호환 API와 MLflow 살펴보기

학생과 교수를 위한 AI 어시스턴트라는 RDU의 아이디어가 구체화되기 시작했습니다. 더 복잡한 사용 사례로 넘어가기 전에, 내부에서 어떤 일이 일어나고 있는지 살펴보겠습니다.

이 노트북에서는 현재 Canopy 애플리케이션의 두 가지 핵심 구성 요소를 살펴봅니다.

- **OpenAI 호환 클라이언트**: 모델에 프롬프트를 보내는 데 사용합니다
- **MLflow**: 프롬프트를 관리하고 상호작용을 추적(trace)하는 데 사용합니다

직접 실습해 보겠습니다!

![](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExNnRhZXg4eDlpZG83NnloeWJ6aGw3ZGp0bWl1azdobmNza2dobTNuYyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/uENJtJp1SGd597E7tL/giphy.gif)

## 1. 모델에 연결하기

Canopy 개발자인 우리에게는 모델 접근 권한이 주어져 있습니다. 모델 서버는 **OpenAI 호환 API**를 제공하므로 표준 `openai` Python 라이브러리로 모델과 통신할 수 있습니다.

모델의 엔드포인트만 지정하면 됩니다.

In [ ]:
from openai import OpenAI

LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #일단은
)

## 2. 시스템 프롬프트 작성하기

앞에서 만든 시스템 프롬프트를 가져와 아래 셀을 수정하고 실험을 시작해 보세요.

In [ ]:
sys_prompt = """You are a helpful assistant. Summarize this:"""

#### Canopy에 관한 사용자 메시지 예시입니다. 요약해 보겠습니다!

In [ ]:
user_message = """
Canopy (Biology)

In biology and ecology, the canopy refers to the upper layer or \"roof\" formed by the crowns of trees in a forest or wooded area. This layer plays a critical role in regulating the ecosystem by controlling light penetration, humidity, temperature, and wind flow within the forest environment. The canopy is typically made up of the tallest trees and their branches and leaves, which often form a dense, continuous cover that can be several meters thick.

One of the primary ecological functions of the canopy is to provide habitat and food sources for a wide range of organisms. Many species of birds, insects, mammals, and epiphytes (plants that grow on other plants) are specially adapted to live in this elevated environment. The canopy also acts as a barrier that reduces the impact of heavy rain on the forest floor, helping to prevent soil erosion and maintain soil fertility.

Moreover, the canopy plays a crucial role in photosynthesis on a large scale by capturing sunlight and converting it into chemical energy, which sustains the forest's plant life and, consequently, the animals that depend on it. In tropical rainforests, the canopy is often so dense that very little sunlight reaches the forest floor, shaping the types of plants and animals that can survive in the understory and ground layers.

Scientists study canopies using specialized tools and methods such as canopy cranes, drones, and climbing equipment to better understand their structure, biodiversity, and ecological functions. This knowledge is vital for conservation efforts, particularly as canopies are sensitive to deforestation, climate change, and human activities that threaten their integrity.

Understanding the canopy's complexity helps ecologists appreciate the interdependent relationships within forests and the critical services these ecosystems provide, including carbon storage, oxygen production, and climate regulation. Protecting the canopy is essential to maintaining biodiversity and the health of our planet.

"""

## 3. 요청 보내기

스트리밍 chat completion 요청을 보내 보겠습니다. OpenAI를 사용할 때와 같은 형태의 API이며, 유일한 차이점은 `base_url`이 자체 호스팅 모델을 가리킨다는 것입니다.

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

모델 서버가 OpenAI 프로토콜을 사용하므로 애플리케이션 코드를 바꾸지 않고도 모델을 교체할 수 있습니다. 엔드포인트와 모델 이름만 업데이트하면 됩니다. 모델별 로직은 필요하지 않습니다!

---
## 4. MLflow 사용하기

이제 **MLflow**를 활용해 보겠습니다. 현재 MLflow는 다음과 같은 일을 도와줍니다.

- 버전이 관리되는 레지스트리에서 **프롬프트를 관리**하여 어떤 프롬프트가 효과적이었는지 놓치지 않게 해 줍니다

먼저 MLflow 서버에 대한 연결을 설정하겠습니다.

In [ ]:
import os
import mlflow
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("canopy-experiment")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

## 5. 프롬프트 레지스트리

앞에서 MLflow Prompt Registry에 저장한 프롬프트를 기억하시나요? 여기에서 가져와 보겠습니다!

아래 `PROMPT_NAME`을 레지스트리에서 프롬프트를 만들 때 사용한 이름으로 수정하세요. 예를 들어 `summarization`이라는 이름을 사용했다면 그 이름을 입력합니다.

In [ ]:
PROMPT_NAME = "summarization"

In [ ]:
prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")

print("Loaded prompt from registry:")
print(prompt.template)

In [ ]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

이제 프롬프트는 코드에 하드코딩되지 않고 레지스트리에 저장되어 있습니다. 프롬프트에 버전과 태그를 붙이고 여러 애플리케이션에서 공유할 수 있으며, 프롬프트를 업데이트할 때 코드를 바꿀 필요가 없습니다!

## 6. 새 프롬프트 버전 저장하기

프롬프트 버전은 코드로도 관리할 수 있습니다. 다른 템플릿으로 프롬프트의 새 버전을 등록하고 태그를 붙여 보겠습니다.

In [ ]:
new_prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="You are a concise academic assistant. Summarize the following text in 3 bullet points:",
    commit_message="Shorter bullet-point style summarization prompt",
    tags={"style": "bullet-points"},
)

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=new_prompt.version)

print(f"Registered prompt: {new_prompt.name}")
print(f"Version: {new_prompt.version}")
print(f"Alias: champion")
print(f"Template:\n{new_prompt.template}")

OpenShift AI 대시보드로 이동해 확인해 보세요. 태그와 `champion` 별칭(alias)이 붙은 새 버전의 프롬프트가 보일 것입니다. 같은 이름으로 `register_prompt`를 호출하면 새 버전이 자동으로 생성됩니다!

![mlflow-champion-prompt.png](./mlflow-champion-prompt.png)

이제 별칭으로 프롬프트를 불러와 사용해 보겠습니다.

In [ ]:
champion_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": champion_prompt.template},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)

for chunk in response:
    if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
        delta = chunk.choices[0].delta
        if hasattr(delta, 'content') and delta.content:
            print(delta.content, end="", flush=True)
print()

---

실습 안내 문서로 돌아가 지금까지 다룬 내용을 Canopy 백엔드에 적용하고, LLM 로직을 프론트엔드에서 분리해 보세요!